# Vyuha P11 - L3 vs AgentDojo (indirect prompt-injection defense)

Runs the **AgentDojo** benchmark (Debenedetti et al., NeurIPS 2024) with a **free Gemini AI-Studio key** as the agent, comparing the **undefended** agent against the agent behind our **L3 injection defense** (`InjectionScanner`, which de-obfuscates each tool output and sanitizes injected instruction sentences).

Reports **utility-under-attack** and **injection ASR** for each. `security = fraction of injections that FAILED`; `ASR = 1 - security`.

**Setup:** Settings -> Internet: **ON**. Add your AI-Studio key as a Kaggle **Secret** named `GEMINI_API_KEY`. No GPU needed (Gemini runs remotely). Free tier is ~1500 req/day, so this runs a **subset**; `logdir` caches results, so a run cut off by the daily cap **resumes** next time.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/vyuha.git"
DEST = "/kaggle/working/vyuha_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/vyuha/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "vyuha" or m.startswith(("vyuha.", "eval"))]:
    del sys.modules[m]
subprocess.run("pip -q install agentdojo 2>/dev/null", shell=True)
print("vyuha repo at:", root)

## Run: undefended vs Vyuha L3

Pulls `GEMINI_API_KEY` from Kaggle Secrets. Defaults: **banking** suite, 4 user tasks x 2 injection tasks, `important_instructions` attack, `gemini-2.0-flash-001`. Increase `n_user_tasks` / `n_injection_tasks` for more coverage (watch the daily quota - the cell backs off automatically on 429s).

In [ ]:
from kaggle_secrets import UserSecretsClient
api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")

from eval.agentdojo_eval import run_agentdojo_l3
# Free-tier friendly defaults: flash-lite (~1000 req/day, its own quota bucket) + a small
# subset. Bump n_user_tasks / n_injection_tasks if you have daily quota to spare.
results = run_agentdojo_l3(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    suite_name="banking",
    n_user_tasks=2,
    n_injection_tasks=1,
)
results